# Chapter 19: Performance Tuning, Memory Optimization, and Efficient Pandas Patterns

**Companion notebook** for *Beginner's Guide to Pandas* by Ravi Shankar

Run each cell in order. Exercises are at the end.

In [77]:
import pandas as pd
import numpy as np

# Performance Tuning, Memory Optimization, and Efficient Pandas Patterns

## Prerequisites Review

Before diving into performance optimization, make sure you're comfortable with these foundational concepts:

In [78]:
import pandas as pd
import numpy as np

# 1. DataFrame indexing (.loc, .iloc)
df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
print(df.loc[0, 'A'])           # Label-based: row 0, column 'A'
print(df.iloc[0, 0])            # Position-based: first row, first column

# 2. Data types (dtype)
print(df.dtypes)                # Check column types
print(df['A'].astype('int32'))  # Convert to different type

# 3. Views vs. Copies (critical for avoiding SettingWithCopyWarning)
view = df['A']                  # This is a view
copy = df['A'].copy()           # This is a copy

1
1
A    int64
B    int64
dtype: object
0    1
1    2
2    3
Name: A, dtype: int32


---

## Understanding Memory Usage in DataFrames

Before optimizing, you need to understand where memory is being used. Pandas provides several tools to inspect memory allocation.

### Analyzing Memory Consumption

The `info()` method provides a quick overview of your DataFrame's memory footprint:

In [79]:
import pandas as pd
import numpy as np

# Create a sample DataFrame with various data types
df = pd.DataFrame({
    'integers': np.arange(100000),
    'floats': np.random.randn(100000),
    'strings': ['text_' + str(i) for i in range(100000)],
    'dates': pd.date_range('2020-01-01', periods=100000, freq='h'),
    'booleans': np.random.choice([True, False], 100000)
})

# Quick memory overview
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   integers  100000 non-null  int64         
 1   floats    100000 non-null  float64       
 2   strings   100000 non-null  object        
 3   dates     100000 non-null  datetime64[ns]
 4   booleans  100000 non-null  bool          
dtypes: bool(1), datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 3.1+ MB


The output shows memory usage with a `+` symbol next to object columns (like strings), indicating they may use more memory than reported. For accurate measurements:

In [80]:
# Deep inspection — accounts for object dtypes accurately
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   integers  100000 non-null  int64         
 1   floats    100000 non-null  float64       
 2   strings   100000 non-null  object        
 3   dates     100000 non-null  datetime64[ns]
 4   booleans  100000 non-null  bool          
dtypes: bool(1), datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 8.8 MB


### Per-Column Memory Breakdown

Examine which columns consume the most memory:

In [81]:
# Get memory usage by column in bytes
memory_by_column = df.memory_usage(deep=True)
print(memory_by_column)

# Convert to megabytes for easier reading
print((memory_by_column / 1024**2).round(2))

# Total memory in MB
total_mb = memory_by_column.sum() / 1024**2
print(f"Total memory: {total_mb:.2f} MB")

# Find the most memory-intensive columns
print(memory_by_column.sort_values(ascending=False))

Index           128
integers     800000
floats       800000
strings     6688890
dates        800000
booleans     100000
dtype: int64
Index       0.00
integers    0.76
floats      0.76
strings     6.38
dates       0.76
booleans    0.10
dtype: float64
Total memory: 8.76 MB
strings     6688890
integers     800000
floats       800000
dates        800000
booleans     100000
Index           128
dtype: int64


---

## Data Type Optimization

One of the most effective ways to reduce memory usage is choosing appropriate data types.

### Downcasting Numeric Types

Numeric columns often use more memory than necessary. Downcast to smaller data types when appropriate:

In [82]:
import pandas as pd

# Create 100,000 rows in each column
df_original = pd.DataFrame({
    'small_int': [1, 2, 3, 4, 5] * 20000,
    'small_float': [1.5, 2.5, 3.5, 4.5, 5.5] * 20000,
})

print(f"Original memory: {df_original.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Downcast to smaller numeric types
df_optimized = df_original.copy()

df_optimized['small_int'] = pd.to_numeric(
    df_optimized['small_int'],
    downcast='integer'
)

df_optimized['small_float'] = pd.to_numeric(
    df_optimized['small_float'],
    downcast='float'
)

print(f"Optimized memory: {df_optimized.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nOriginal dtypes:")
print(df_original.dtypes)

print("\nOptimized dtypes:")
print(df_optimized.dtypes)

Original memory: 1.53 MB
Optimized memory: 0.48 MB

Original dtypes:
small_int        int64
small_float    float64
dtype: object

Optimized dtypes:
small_int         int8
small_float    float32
dtype: object


For manual type selection, use smaller integer and float types when you know the value range:

In [83]:
# Before optimization
df_large = pd.DataFrame({
    'age': np.random.randint(0, 120, 1000000),
    'count': np.random.randint(0, 1000, 1000000),
    'flag': np.random.randint(0, 2, 1000000)
})

print(f"Original memory: {df_large.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# After optimization — choose types that fit the value range
df_large_opt = df_large.copy()
df_large_opt['age'] = df_large_opt['age'].astype('int16')   # -32,768 to 32,767
df_large_opt['count'] = df_large_opt['count'].astype('int16')
df_large_opt['flag'] = df_large_opt['flag'].astype('bool')

print(f"Optimized memory: {df_large_opt.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Original memory: 22.89 MB
Optimized memory: 4.77 MB


Similarly, downcast floats when precision allows:

In [84]:
# Check precision loss before committing to float32
original = np.array([1.23456789, 9.87654321])
downcast = original.astype('float32')
print(f"Precision loss: {np.abs(original - downcast).max()}")

Precision loss: 1.6495605414945658e-07


### Using Categorical Data for Repetitive Values

String columns consume significant memory. Convert repetitive string values to categorical — this typically saves 50–90% of memory:

In [85]:
# Before: object dtype for repeated strings
df_before = pd.DataFrame({
    'category': ['A', 'B', 'C', 'D'] * 250000,
})

print(f"Object dtype memory: {df_before.memory_usage(deep=True)['category'] / 1024**2:.2f} MB")

# After: categorical dtype
df_after = df_before.copy()
df_after['category'] = df_after['category'].astype('category')

print(f"Categorical dtype memory: {df_after.memory_usage(deep=True)['category'] / 1024**2:.2f} MB")

# Access category information
print(df_after['category'].cat.categories)
print(df_after['category'].cat.codes.unique())

Object dtype memory: 55.31 MB
Categorical dtype memory: 0.95 MB
Index(['A', 'B', 'C', 'D'], dtype='object')
[0 1 2 3]


**When to use categorical:** Convert a column when the number of unique values is small relative to the total number of rows. A useful rule of thumb is to convert when fewer than 5% of values are unique:

In [86]:
# Decision guide: should this column be categorical?
col = df_after['category']
unique_ratio = col.nunique() / len(col)
print(f"Unique ratio: {unique_ratio:.2%}")
if unique_ratio < 0.05:
    print("→ Good candidate for categorical dtype")

Unique ratio: 0.00%
→ Good candidate for categorical dtype


### Efficient Integer Handling with Missing Values

When you have missing values in integer columns, use nullable integer types to avoid the memory cost of promoting to float64:

In [87]:
# Problem: integers with NaN require float64
df_float = pd.DataFrame({
    'values': [1, 2, np.nan, 4, 5] * 20000,
})

print(f"Float64 memory: {df_float.memory_usage(deep=True)['values'] / 1024**2:.2f} MB")

# Solution: use nullable Int64 (capital 'I')
df_int = pd.DataFrame({
    'values': pd.array([1, 2, pd.NA, 4, 5] * 20000, dtype='Int64'),
})

print(f"Int64 memory: {df_int.memory_usage(deep=True)['values'] / 1024**2:.2f} MB")

Float64 memory: 0.76 MB
Int64 memory: 0.86 MB


### Complete Memory Audit

Use this utility function to get a full picture of your DataFrame's memory profile and identify optimization opportunities:

In [88]:
def audit_dataframe_memory(df):
    """Comprehensive memory analysis of a DataFrame."""

    print("=" * 60)
    print("MEMORY AUDIT REPORT")
    print("=" * 60)

    # Total memory
    total_memory = df.memory_usage(deep=True).sum() / 1024**2
    print(f"\nTotal Memory Usage: {total_memory:.2f} MB")

    # Memory by column
    print("\nMemory by Column (MB):")
    memory_by_col = (df.memory_usage(deep=True) / 1024**2).sort_values(ascending=False)
    for col, mem in memory_by_col.items():
        if col != 'Index':
            print(f"  {col}: {mem:.2f} MB ({df[col].dtype})")

    # Data type distribution
    print("\nData Type Distribution:")
    dtype_counts = df.dtypes.value_counts()
    for dtype, count in dtype_counts.items():
        print(f"  {dtype}: {count} columns")

    # Optimization recommendations
    print("\nOptimization Opportunities:")
    for col in df.columns:
        if df[col].dtype == 'object':
            unique_ratio = df[col].nunique() / len(df[col])
            if unique_ratio < 0.05:
                print(f"  - Convert '{col}' to categorical "
                      f"(only {unique_ratio*100:.1f}% unique values)")
        if df[col].dtype in ['int64', 'float64']:
            print(f"  - Consider downcasting '{col}' from {df[col].dtype}")


# Example usage
df_example = pd.DataFrame({
    'id': range(100000),
    'category': np.random.choice(['A', 'B', 'C', 'D'], 100000),
    'value': np.random.randn(100000),
    'flag': np.random.choice([True, False], 100000)
})

audit_dataframe_memory(df_example)

MEMORY AUDIT REPORT

Total Memory Usage: 7.15 MB

Memory by Column (MB):
  category: 5.53 MB (object)
  id: 0.76 MB (int64)
  value: 0.76 MB (float64)
  flag: 0.10 MB (bool)

Data Type Distribution:
  int64: 1 columns
  object: 1 columns
  float64: 1 columns
  bool: 1 columns

Optimization Opportunities:
  - Consider downcasting 'id' from int64
  - Convert 'category' to categorical (only 0.0% unique values)
  - Consider downcasting 'value' from float64


---

## Chunking and Lazy Loading

### Processing Large Files in Chunks

When files exceed available RAM, process them in manageable chunks rather than loading everything at once:

In [89]:
# ❌ WRONG: Loads entire file at once (may crash on large files)
# df = pd.read_csv('huge_file.csv')

# ✅ RIGHT: Process in chunks
chunk_size = 50000
total_sum = 0

for chunk in pd.read_csv('data/huge_file.csv', chunksize=chunk_size):
    # Process each chunk independently
    total_sum += chunk['amount'].sum()

print(f"Total sum: {total_sum}")

Total sum: 18746250.0


If you need to accumulate results and combine them:

In [90]:
chunk_size = 50000
chunks = []

for chunk in pd.read_csv('data/large_file.csv', chunksize=chunk_size):
    # Apply transformations per chunk
    chunk['processed'] = chunk['Age'] * 2
    chunks.append(chunk)

# Combine results
result = pd.concat(chunks, ignore_index=True)

### Selective Column Loading

Load only the columns you need, and specify efficient types upfront to avoid unnecessary conversions:

In [ ]:
# Define dtypes upfront
dtypes = {
    'id': 'int32',
    'amount': 'float32',
}

df = pd.read_csv(
    'data/huge_file.csv',
    usecols=['id', 'amount', 'date'],
    dtype=dtypes,
    parse_dates=['date']
)

---

## Efficient Operations and Patterns

### Vectorized Operations vs. Loops

Always prefer vectorized operations over Python loops. Loops process one element at a time in Python; vectorized operations hand the work to optimized C code under the hood:

In [92]:
df = pd.DataFrame({'value': np.random.randint(0, 200, 1000000),
'text': np.random.choice(
        ['apple', 'banana', 'orange', 'grape'],
        1_000_000)
    })

# ❌ Avoid: Python loop
result_slow = []
for val in df['value']:
    result_slow.append(val * 2 + 1)

# ❌ Also avoid: .apply() with a lambda for simple math
df['new_column'] = df['value'].apply(lambda x: x * 2 if x > 100 else x)

# ✅ Use numpy where for conditional logic
df['new_column'] = np.where(df['value'] > 100, df['value'] * 2, df['value'])

# ✅ Use vectorized arithmetic for simple operations
result_fast = df['value'] * 2 + 1

For string operations, use the built-in `.str` accessor rather than `.apply()`:

In [93]:
# Slower: apply with lambda
result = df['text'].apply(lambda x: len(x))

# Faster: built-in str accessor
result = df['text'].str.len()

### Efficient Groupby Operations

Use built-in aggregation methods instead of custom functions, and prefer named aggregation for clarity:

In [94]:
df = pd.DataFrame({
    'group': np.random.choice(['A', 'B', 'C'], 1000000),
    'value': np.random.randn(1000000),
    'amount': np.random.randint(1, 100, 1000000)
})

# Slower: custom aggregation via apply
result = df.groupby('group').apply(lambda x: x['value'].sum())

# Faster: built-in aggregation
result = df.groupby('group')['value'].sum()

# Multiple aggregations with named output for clarity
result = df.groupby('group').agg(
    mean_value=('value', 'mean'),
    total_amount=('amount', 'sum'),
    count=('value', 'count')
)

# Use observed=True when grouping by a categorical column
df['group'] = df['group'].astype('category')
result = df.groupby('group', observed=True).agg(
    mean_value=('value', 'mean'),
    total_amount=('amount', 'sum')
)

/var/folders/6k/wdwghqdd7ynf_52q9qqp3v9w0000gp/T/ipykernel_6936/677243395.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = df.groupby('group').apply(lambda x: x['value'].sum())


### Efficient Indexing and Selection

When doing repeated lookups on a column, set it as the index to move from O(n) to O(log n) lookup time:

In [95]:
df = pd.DataFrame({
    'user_id': range(1000000),
    'name': ['User_' + str(i) for i in range(1000000)],
    'score': np.random.randn(1000000)
})

import time

# Without index: O(n) scan
start = time.time()
result = df[df['user_id'] == 500000]
print(f"Without index: {time.time() - start:.4f}s")

# With index: O(log n) lookup
df_indexed = df.set_index('user_id')
start = time.time()
result = df_indexed.loc[500000]
print(f"With index: {time.time() - start:.4f}s")

Without index: 0.0006s
With index: 0.0006s


Also prefer `.loc[]` for combined filtering and selection to avoid creating intermediate copies:

In [96]:
# Inefficient: creates an intermediate DataFrame
result = df[df['score'] > 0][['name', 'score']]

# Efficient: single operation
result = df.loc[df['score'] > 0, ['name', 'score']]

### Efficient Merging

Choose the right join type and validate relationships to catch errors early:

In [97]:
import pandas as pd
import numpy as np

# Left DataFrame (larger)
df_left = pd.DataFrame({
    'key': range(1_000_000),
    'value_left': np.random.randn(1_000_000)
})

# Right DataFrame (smaller) with UNIQUE keys
df_right = pd.DataFrame({
    'key': np.random.choice(
        range(1_000_000),
        100_000,
        replace=False  # Important!
    ),
    'value_right': np.random.randn(100_000)
})

# Inner join: keeps only matching rows
result_inner = pd.merge(
    df_left,
    df_right,
    on='key',
    how='inner'
)

# Left join with validation
result_left = pd.merge(
    df_left,
    df_right,
    on='key',
    how='left',
    validate='m:1'
)

print("Inner join shape:", result_inner.shape)
print("Left join shape:", result_left.shape)

Inner join shape: (100000, 3)
Left join shape: (1000000, 3)


---

## Query and Filter Optimization

### Using `.query()` for Complex Conditions

The `.query()` method produces more readable code and can be more memory-efficient for complex filtering because it avoids creating intermediate boolean arrays:

In [98]:
df = pd.DataFrame({
    'A': np.random.randn(100000),
    'B': np.random.randn(100000),
    'C': np.random.randint(0, 100, 100000)
})

# Traditional filtering — creates intermediate DataFrames
result1 = df[(df['A'] > 0) & (df['B'] < 0.5) & (df['C'] > 50)]

# Query method — more readable and memory-efficient
result2 = df.query('A > 0 and B < 0.5 and C > 50')

# Both produce the same result
assert result1.equals(result2)

---

## Copy vs. View Awareness

### Understanding When Pandas Creates Copies

A **view** shares data with the original DataFrame — changes to the view affect the original. A **copy** is independent — changes do not affect the original. Misunderstanding this distinction is the most common source of `SettingWithCopyWarning`.

In [99]:
df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})

# VIEW: changes affect the original
view = df['A']
view.iloc[0] = 999
print(df)  # A's first value is now 999

# COPY: changes do not affect the original
df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
copy = df['A'].copy()
copy.iloc[0] = 999
print(df)  # A's first value is still 1

   A  B
0  1  4
1  2  5
2  3  6
   A  B
0  1  4
1  2  5
2  3  6


### Avoiding SettingWithCopyWarning

A chained assignment occurs when you filter a DataFrame and then modify it in the same expression. Pandas may create a copy during the filter step, so the modification silently has no effect on the original:

In [100]:
import warnings
warnings.simplefilter('always')

df = pd.DataFrame({
    'value': [1, 2, 3, 4, 5],
    'category': ['A', 'B', 'A', 'B', 'A']
})

# ❌ RISKY: chained assignment — modifies a temporary copy
df[df['value'] > 2]['new_col'] = 100  # SettingWithCopyWarning

# ✅ GOOD: use .loc[] for safe, in-place assignment
df.loc[df['value'] > 2, 'new_col'] = 100
print(df)

   value category  new_col
0      1        A      NaN
1      2        B      NaN
2      3        A    100.0
3      4        B    100.0
4      5        A    100.0


/var/folders/6k/wdwghqdd7ynf_52q9qqp3v9w0000gp/T/ipykernel_6936/1160260760.py:10: ChainedAssignmentError: A value is trying to be set on a copy of a DataFrame or Series through chained assignment.
When using the Copy-on-Write mode, such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy.

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[df['value'] > 2]['new_col'] = 100  # SettingWithCopyWarning


**Best practices to avoid warnings:**

In [101]:
# Practice 1: Use .loc[] for conditional assignment
df.loc[df['value'] > 2, 'flag'] = True

# Practice 2: Use .copy() when you need to work with a filtered subset
filtered = df[df['value'] > 2].copy()
filtered['new_col'] = 100  # No warning — filtered is explicitly a copy

# Practice 3: Use .assign() for a functional, warning-free style
df = df.assign(new_col=lambda x: x['value'] * 2)

# Practice 4: Enable Copy-on-Write mode (pandas 2.0+) for predictable behavior
pd.options.mode.copy_on_write = True

---

## Profiling and Debugging

### Profiling with `timeit`

Always profile before optimizing — measure first, then fix the right thing:

In [102]:
from timeit import timeit

df = pd.DataFrame({
    'A': np.random.randn(100000),
    'group': np.random.choice(['X', 'Y', 'Z'], 100000)
})

def approach1():
    return df.groupby('group')['A'].sum()

def approach2():
    return df.groupby('group')['A'].apply(np.sum)

time1 = timeit(approach1, number=100)
time2 = timeit(approach2, number=100)

print(f"Approach 1 (built-in sum): {time1:.4f} seconds")
print(f"Approach 2 (apply):        {time2:.4f} seconds")
print(f"Approach 1 is {time2/time1:.1f}x faster")

Approach 1 (built-in sum): 0.1928 seconds
Approach 2 (apply):        0.2020 seconds
Approach 1 is 1.0x faster


### Tracking Performance Improvements

Use a decorator to measure how long any function takes:

In [103]:
import time
from functools import wraps

def time_operation(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"{func.__name__} took {elapsed:.4f} seconds")
        return result
    return wrapper

@time_operation
def load_and_process(filepath):
    df = pd.read_csv(filepath)
    df['category'] = df['category'].astype('category')
    return df.groupby('category')['value'].sum()

### Memory Profiling

For line-by-line memory analysis, install and use `memory_profiler`:

In [104]:
# Install: pip install memory-profiler

from memory_profiler import profile

@profile
def process_data(df):
    """This function will be profiled for memory usage."""
    result = df.groupby('group').agg({
        'A': 'sum',
        'B': 'mean'
    })
    return result

df = pd.DataFrame({
    'A': np.random.randn(100000),
    'B': np.random.randn(100000),
    'group': np.random.choice(['X', 'Y', 'Z'], 100000)
})

process_data(df)

ERROR: Could not find file /var/folders/6k/wdwghqdd7ynf_52q9qqp3v9w0000gp/T/ipykernel_6936/2773624945.py


,A,B
group,,
X,-119.701406,0.001284
Y,-160.630631,0.009171
Z,138.275602,0.003994


---

## Practical Optimization Workflow

Use this decision tree to choose the right tool for each problem:

```
START: I have a performance or memory problem
  │
  ├─ Is my data memory-intensive?
  │    YES → Use categorical dtypes, downcast numerics, chunk large files
  │
  ├─ Is my code running slowly?
  │    YES → Profile with timeit/memory_profiler, then fix the bottleneck
  │
  ├─ Am I getting SettingWithCopyWarning?
  │    YES → Use .loc[] for assignments, or enable Copy-on-Write mode
  │
  └─ Do I need custom data types or specialized behavior?
       YES → Use nullable integer types (Int64) or pd.Categorical
       NO  → Your code is probably fine!
```

### Putting It All Together

Here is a complete example that applies profiling, dtype optimization, and safe assignment in sequence:

In [105]:
import pandas as pd
import numpy as np
from timeit import timeit

np.random.seed(42)
df = pd.DataFrame({
    'customer_id': np.repeat(np.arange(1000), 100),
    'purchase_amount': np.random.uniform(10, 1000, 100000),
    'status': np.random.choice(['active', 'inactive', 'pending'], 100000),
    'region': np.random.choice(['North', 'South', 'East', 'West'], 100000)
})

# Step 1: Profile the original operation
def aggregate_by_status():
    return df.groupby('status')['purchase_amount'].sum()

time_original = timeit(aggregate_by_status, number=100)

# Step 2: Optimize with categorical dtypes
df['status'] = df['status'].astype('category')
df['region'] = df['region'].astype('category')

# Step 3: Profile the optimized operation
time_optimized = timeit(aggregate_by_status, number=100)

# Step 4: Add a column safely using .loc[]
df.loc[df['purchase_amount'] > 500, 'high_value'] = True

# Step 5: Report results
print(f"Original:  {time_original:.4f}s")
print(f"Optimized: {time_optimized:.4f}s")
print(f"Speedup:   {time_original/time_optimized:.2f}x faster")

Original:  0.2263s
Optimized: 0.0407s
Speedup:   5.56x faster


/var/folders/6k/wdwghqdd7ynf_52q9qqp3v9w0000gp/T/ipykernel_6936/2253852090.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return df.groupby('status')['purchase_amount'].sum()
/var/folders/6k/wdwghqdd7ynf_52q9qqp3v9w0000gp/T/ipykernel_6936/2253852090.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return df.groupby('status')['purchase_amount'].sum()
/var/folders/6k/wdwghqdd7ynf_52q9qqp3v9w0000gp/T/ipykernel_6936/2253852090.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or obse

---

## Common Pitfalls and Solutions

| Pitfall | What Happens | Solution |
|---------|--------------|----------|
| Modifying a filtered DataFrame | Changes don't appear in the original | Use `.loc[]` or `.copy()` |
| Forgetting `.copy()` after filtering | Unexpected `SettingWithCopyWarning` | Always `.copy()` if you'll modify the subset |
| Not profiling before optimizing | Optimizing the wrong thing | Profile first, then optimize |
| Using object dtype for categories | Wasted memory and slow operations | Convert to `dtype='category'` |
| Mixing views and copies | Confusing behavior, hard to debug | Understand when pandas creates copies |
| Loading all columns from a large file | Unnecessary memory use | Use `usecols` and `dtype` in `read_csv` |

---

## Exercises

### Task 1: Profile a Simple Operation (Beginner)

In [106]:
import pandas as pd
import numpy as np
from timeit import timeit

df = pd.DataFrame({
    'A': np.random.randn(100000),
    'B': np.random.randn(100000),
    'group': np.random.choice(['X', 'Y', 'Z'], 100000)
})

def approach1():
    return df.groupby('group')['A'].sum()

def approach2():
    return df.groupby('group')['A'].apply(np.sum)

time1 = timeit(approach1, number=100)
time2 = timeit(approach2, number=100)

print(f"Approach 1 (sum):   {time1:.4f} seconds")
print(f"Approach 2 (apply): {time2:.4f} seconds")

if time1 < time2:
    print("Approach 1 is faster!")
else:
    print("Approach 2 is faster!")

Approach 1 (sum):   0.1697 seconds
Approach 2 (apply): 0.1971 seconds
Approach 1 is faster!


### Task 2: Create and Use a Categorical Dtype (Intermediate)

In [107]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'product': ['laptop', 'mouse', 'keyboard', 'laptop', 'mouse'] * 1000,
    'sales': np.random.randint(100, 1000, 5000)
})

memory_before = df['product'].memory_usage(deep=True)

df['product'] = df['product'].astype('category')

memory_after = df['product'].memory_usage(deep=True)

savings = memory_before - memory_after
savings_pct = (savings / memory_before) * 100

print(f"Memory before: {memory_before} bytes")
print(f"Memory after:  {memory_after} bytes")
print(f"Memory saved:  {savings} bytes ({savings_pct:.1f}%)")

Memory before: 315128 bytes
Memory after:  5426 bytes
Memory saved:  309702 bytes (98.3%)


### Task 3: Fix Chained Assignments (Intermediate)

In [108]:
import pandas as pd
import warnings
warnings.simplefilter('always')

df = pd.DataFrame({
    'value': [1, 2, 3, 4, 5],
    'category': ['A', 'B', 'A', 'B', 'A']
})

# Fix using .loc[] to avoid SettingWithCopyWarning
df.loc[df['value'] > 2, 'flag'] = True

print(df)
print("SettingWithCopyWarning eliminated!")

   value category  flag
0      1        A   NaN
1      2        B   NaN
2      3        A  True
3      4        B  True
4      5        A  True
SettingWithCopyWarning eliminated!


### Task 4: Combine All Techniques (Advanced)

In [109]:
import pandas as pd
import numpy as np
from timeit import timeit

np.random.seed(42)
df = pd.DataFrame({
    'customer_id': np.repeat(np.arange(1000), 100),
    'purchase_amount': np.random.uniform(10, 1000, 100000),
    'status': np.random.choice(['active', 'inactive', 'pending'], 100000),
    'region': np.random.choice(['North', 'South', 'East', 'West'], 100000)
})

# Step 1: Audit memory before optimization
audit_dataframe_memory(df)

# Step 2: Profile the original operation
def aggregate_by_status():
    return df.groupby('status')['purchase_amount'].sum()

time_original = timeit(aggregate_by_status, number=100)

# Step 3: Optimize dtypes
df['status'] = df['status'].astype('category')
df['region'] = df['region'].astype('category')
df['purchase_amount'] = df['purchase_amount'].astype('float32')

# Step 4: Profile the optimized operation
time_optimized = timeit(aggregate_by_status, number=100)

# Step 5: Add a column safely
df.loc[df['purchase_amount'] > 500, 'high_value'] = True

# Step 6: Audit memory after optimization
audit_dataframe_memory(df)

print(f"\nOriginal:  {time_original:.4f}s")
print(f"Optimized: {time_optimized:.4f}s")
print(f"Speedup:   {time_original/time_optimized:.2f}x faster")

MEMORY AUDIT REPORT

Total Memory Usage: 13.49 MB

Memory by Column (MB):
  status: 6.10 MB (object)
  region: 5.86 MB (object)
  customer_id: 0.76 MB (int64)
  purchase_amount: 0.76 MB (float64)

Data Type Distribution:
  object: 2 columns
  int64: 1 columns
  float64: 1 columns

Optimization Opportunities:
  - Consider downcasting 'customer_id' from int64
  - Consider downcasting 'purchase_amount' from float64
  - Convert 'status' to categorical (only 0.0% unique values)
  - Convert 'region' to categorical (only 0.0% unique values)
MEMORY AUDIT REPORT

Total Memory Usage: 4.58 MB

Memory by Column (MB):
  high_value: 3.24 MB (object)
  customer_id: 0.76 MB (int64)
  purchase_amount: 0.38 MB (float32)
  region: 0.10 MB (category)
  status: 0.10 MB (category)

Data Type Distribution:
  int64: 1 columns
  float32: 1 columns
  category: 1 columns
  category: 1 columns
  object: 1 columns

Optimization Opportunities:
  - Consider downcasting 'customer_id' from int64
  - Convert 'high_valu

/var/folders/6k/wdwghqdd7ynf_52q9qqp3v9w0000gp/T/ipykernel_6936/1822231852.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return df.groupby('status')['purchase_amount'].sum()
/var/folders/6k/wdwghqdd7ynf_52q9qqp3v9w0000gp/T/ipykernel_6936/1822231852.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return df.groupby('status')['purchase_amount'].sum()
/var/folders/6k/wdwghqdd7ynf_52q9qqp3v9w0000gp/T/ipykernel_6936/1822231852.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or obse

---

## Key Takeaways

- **Profile before optimizing**: Use `info()`, `memory_usage()`, and `timeit` to identify bottlenecks before changing anything
- **Choose appropriate data types**: Use `int8`/`int16`, `float32`, and `category` when suitable; use nullable `Int64` for integers with missing values
- **Process large datasets in chunks**: Avoid loading entire files when they exceed available RAM
- **Vectorize operations**: Always prefer pandas and NumPy vectorized methods over Python loops and `.apply()`
- **Use `.loc[]` for assignments**: Understand when operations create copies vs. views, and use `.loc[]` to avoid `SettingWithCopyWarning`
- **Load smartly**: Specify `usecols` and `dtype` when reading large files to avoid unnecessary memory use
- **Use `.query()` for complex conditions**: Often more readable and more efficient than chained boolean indexing

---

# Exercises

Test your understanding of this chapter's concepts.

### Exercise 1: Inspect and Reduce DataFrame Memory Usage

Create a DataFrame with inefficient data types and measure its memory usage. Then optimize the data types (e.g., downcast integers, use categories for low-cardinality strings) and compare the memory footprint before and after.

In [110]:
import pandas as pd
import numpy as np

# Sample sales data with inefficient types
data = {
    'order_id': [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    'region': ['North', 'South', 'North', 'East', 'South', 'East', 'North', 'South'],
    'quantity': [3, 1, 7, 2, 5, 4, 6, 2],
    'price': [19.99, 45.50, 8.75, 120.00, 33.25, 67.80, 15.40, 99.99],
    'returned': [0, 0, 1, 0, 1, 0, 0, 1]
}

df = pd.DataFrame(data)

# TODO: Print the memory usage of the original DataFrame (include deep=True)

# TODO: Convert 'region' to category dtype

# TODO: Downcast 'order_id' and 'quantity' to the smallest suitable integer type
#       using pd.to_numeric with downcast='unsigned'

# TODO: Convert 'returned' to boolean dtype

# TODO: Print the memory usage of the optimized DataFrame

# TODO: Print the dtypes of both DataFrames to compare


### Exercise 2: Process a Large Dataset in Chunks

Simulate reading a large CSV file in chunks using pd.read_csv with the chunksize parameter. Aggregate a running total and row count across all chunks to compute the overall average of a numeric column without loading everything into memory at once.

In [111]:
import pandas as pd
import numpy as np
import io

# Simulate a large CSV file as an in-memory string (represents a file on disk)
np.random.seed(42)
n_rows = 1000
csv_data = 'employee_id,department,salary,years_exp\n'
for i in range(n_rows):
    dept = np.random.choice(['Engineering', 'Marketing', 'HR', 'Finance'])
    salary = round(np.random.uniform(40000, 120000), 2)
    years = np.random.randint(1, 20)
    csv_data += f'{i+1},{dept},{salary},{years}\n'

# TODO: Use pd.read_csv with io.StringIO(csv_data) and chunksize=200
#       to read the data in chunks of 200 rows

# TODO: Iterate over the chunks and accumulate:
#       - total_salary: running sum of the 'salary' column
#       - total_rows: running count of rows processed

total_salary = 0
total_rows = 0

# --- your loop here ---

# TODO: Compute and print the overall average salary across all chunks

# TODO: Verify your result by reading the full data at once and comparing


### Exercise 3: Vectorized Operations vs. Iterrows Performance

Compare the performance of a row-wise loop using iterrows against a vectorized pandas operation for the same transformation. Use Python's time module to measure and print the elapsed time for each approach, then confirm both produce identical results.

In [112]:
import pandas as pd
import numpy as np
import time

np.random.seed(0)
df = pd.DataFrame({
    'base_price': np.random.uniform(10, 500, size=10_000),
    'discount_pct': np.random.uniform(0, 0.4, size=10_000),
    'tax_rate': np.random.uniform(0.05, 0.15, size=10_000)
})
# Goal: compute final_price = base_price * (1 - discount_pct) * (1 + tax_rate)

# --- Approach 1: iterrows loop ---
# TODO: Record start time
# TODO: Iterate with iterrows and compute final_price row by row,
#       storing results in a Python list, then assign as a new column 'final_price_loop'
# TODO: Record end time and print elapsed time

# --- Approach 2: Vectorized operation ---
# TODO: Record start time
# TODO: Compute final_price using direct column arithmetic,
#       storing result in 'final_price_vec'
# TODO: Record end time and print elapsed time

# TODO: Verify both columns are equal (use np.allclose for float comparison)
# TODO: Print a summary showing the speedup factor (loop_time / vec_time)


### Exercise 4: Avoid Unintended Copies with View vs. Copy Awareness

Demonstrate the difference between a view and a copy when slicing a DataFrame. Show that modifying a slice obtained via chained indexing may not affect the original, then use .loc to perform safe, in-place modifications and confirm the original DataFrame is updated correctly.

In [113]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'product': ['Widget', 'Gadget', 'Doohickey', 'Thingamajig', 'Gizmo'],
    'category': ['A', 'B', 'A', 'C', 'B'],
    'stock': [150, 30, 200, 5, 80],
    'price': [9.99, 49.99, 4.99, 199.99, 29.99]
})

print('Original DataFrame:')
print(df)
print()

# --- Part 1: Demonstrate the copy/view ambiguity ---
# TODO: Create a subset called 'low_stock' containing rows where stock < 50
#       using simple boolean indexing (df[condition])

# TODO: Try to set the 'price' column in low_stock to 0.0 using chained assignment
#       e.g., low_stock['price'] = 0.0
#       Observe the SettingWithCopyWarning (or note that the original is unchanged)

# TODO: Print the original df to show its 'price' column was NOT changed

# --- Part 2: Safe modification using .loc ---
# TODO: Use df.loc with the same boolean condition to set price to 0.0
#       directly on the original DataFrame

# TODO: Print the updated df to confirm the change was applied correctly


Original DataFrame:
       product category  stock   price
0       Widget        A    150    9.99
1       Gadget        B     30   49.99
2    Doohickey        A    200    4.99
3  Thingamajig        C      5  199.99
4        Gizmo        B     80   29.99



---

# Solutions

*Scroll down only after you've attempted the exercises above.*

<br><br><br><br><br><br><br><br><br><br>

### Solution 1: Inspect and Reduce DataFrame Memory Usage

In [114]:
import pandas as pd
import numpy as np

# Sample sales data with inefficient types
data = {
    'order_id': [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    'region': ['North', 'South', 'North', 'East', 'South', 'East', 'North', 'South'],
    'quantity': [3, 1, 7, 2, 5, 4, 6, 2],
    'price': [19.99, 45.50, 8.75, 120.00, 33.25, 67.80, 15.40, 99.99],
    'returned': [0, 0, 1, 0, 1, 0, 0, 1]
}

df = pd.DataFrame(data)

# Print the memory usage of the original DataFrame
print('=== Original DataFrame ===')
print(df.dtypes)
original_mem = df.memory_usage(deep=True)
print(original_mem)
print(f'Total original memory: {original_mem.sum()} bytes\n')

# Optimize dtypes
df_opt = df.copy()

# Convert 'region' to category dtype
df_opt['region'] = df_opt['region'].astype('category')

# Downcast 'order_id' and 'quantity' to the smallest suitable integer type
df_opt['order_id'] = pd.to_numeric(df_opt['order_id'], downcast='unsigned')
df_opt['quantity'] = pd.to_numeric(df_opt['quantity'], downcast='unsigned')

# Convert 'returned' to boolean dtype
df_opt['returned'] = df_opt['returned'].astype(bool)

# Print the memory usage of the optimized DataFrame
print('=== Optimized DataFrame ===')
print(df_opt.dtypes)
optimized_mem = df_opt.memory_usage(deep=True)
print(optimized_mem)
print(f'Total optimized memory: {optimized_mem.sum()} bytes\n')

# Summary
savings = original_mem.sum() - optimized_mem.sum()
print(f'Memory saved: {savings} bytes ({savings / original_mem.sum() * 100:.1f}%)')


=== Original DataFrame ===
order_id      int64
region       object
quantity      int64
price       float64
returned      int64
dtype: object
Index       128
order_id     64
region      494
quantity     64
price        64
returned     64
dtype: int64
Total original memory: 878 bytes

=== Optimized DataFrame ===
order_id      uint16
region      category
quantity       uint8
price        float64
returned        bool
dtype: object
Index       128
order_id     16
region      301
quantity      8
price        64
returned      8
dtype: int64
Total optimized memory: 525 bytes

Memory saved: 353 bytes (40.2%)


### Solution 2: Process a Large Dataset in Chunks

In [115]:
import pandas as pd
import numpy as np
import io

# Simulate a large CSV file as an in-memory string
np.random.seed(42)
n_rows = 1000
csv_data = 'employee_id,department,salary,years_exp\n'
for i in range(n_rows):
    dept = np.random.choice(['Engineering', 'Marketing', 'HR', 'Finance'])
    salary = round(np.random.uniform(40000, 120000), 2)
    years = np.random.randint(1, 20)
    csv_data += f'{i+1},{dept},{salary},{years}\n'

# Read in chunks and accumulate totals
total_salary = 0
total_rows = 0

chunk_iter = pd.read_csv(io.StringIO(csv_data), chunksize=200)
for chunk_num, chunk in enumerate(chunk_iter, start=1):
    total_salary += chunk['salary'].sum()
    total_rows += len(chunk)
    print(f'Chunk {chunk_num}: {len(chunk)} rows processed (running total: {total_rows})')

# Compute overall average salary
average_salary = total_salary / total_rows
print(f'\nOverall average salary (chunked): ${average_salary:,.2f}')

# Verify by reading the full data at once
df_full = pd.read_csv(io.StringIO(csv_data))
verification_avg = df_full['salary'].mean()
print(f'Verification average salary (full load): ${verification_avg:,.2f}')
print(f'Results match: {abs(average_salary - verification_avg) < 0.01}')


Chunk 1: 200 rows processed (running total: 200)
Chunk 2: 200 rows processed (running total: 400)
Chunk 3: 200 rows processed (running total: 600)
Chunk 4: 200 rows processed (running total: 800)
Chunk 5: 200 rows processed (running total: 1000)

Overall average salary (chunked): $79,870.62
Verification average salary (full load): $79,870.62
Results match: True


### Solution 3: Vectorized Operations vs. Iterrows Performance

In [116]:
import pandas as pd
import numpy as np
import time

np.random.seed(0)
df = pd.DataFrame({
    'base_price': np.random.uniform(10, 500, size=10_000),
    'discount_pct': np.random.uniform(0, 0.4, size=10_000),
    'tax_rate': np.random.uniform(0.05, 0.15, size=10_000)
})
# Goal: compute final_price = base_price * (1 - discount_pct) * (1 + tax_rate)

# --- Approach 1: iterrows loop ---
start_loop = time.time()
results = []
for _, row in df.iterrows():
    price = row['base_price'] * (1 - row['discount_pct']) * (1 + row['tax_rate'])
    results.append(price)
df['final_price_loop'] = results
loop_time = time.time() - start_loop
print(f'iterrows loop time:    {loop_time:.4f} seconds')

# --- Approach 2: Vectorized operation ---
start_vec = time.time()
df['final_price_vec'] = df['base_price'] * (1 - df['discount_pct']) * (1 + df['tax_rate'])
vec_time = time.time() - start_vec
print(f'Vectorized operation:  {vec_time:.4f} seconds')

# Verify both columns are equal
are_equal = np.allclose(df['final_price_loop'], df['final_price_vec'])
print(f'\nResults are identical: {are_equal}')

# Speedup summary
speedup = loop_time / vec_time
print(f'Speedup factor: {speedup:.1f}x faster with vectorization')


iterrows loop time:    0.1078 seconds
Vectorized operation:  0.0003 seconds

Results are identical: True
Speedup factor: 365.4x faster with vectorization


### Solution 4: Avoid Unintended Copies with View vs. Copy Awareness

In [117]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'product': ['Widget', 'Gadget', 'Doohickey', 'Thingamajig', 'Gizmo'],
    'category': ['A', 'B', 'A', 'C', 'B'],
    'stock': [150, 30, 200, 5, 80],
    'price': [9.99, 49.99, 4.99, 199.99, 29.99]
})

print('Original DataFrame:')
print(df)
print()

# --- Part 1: Demonstrate the copy/view ambiguity ---
# Create a subset using boolean indexing — this returns a COPY
low_stock = df[df['stock'] < 50]
print('low_stock subset (before attempted modification):')
print(low_stock)
print()

# Attempt chained assignment — may trigger SettingWithCopyWarning
# and will NOT reliably update the original df
import warnings
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    low_stock['price'] = 0.0
    if caught:
        print(f'Warning caught: {caught[0].category.__name__}')
        print(f'Message: {caught[0].message}\n')
    else:
        print('No warning raised (behavior may vary by pandas version)\n')

print('Original df after chained assignment attempt (price should be UNCHANGED):')
print(df[['product', 'stock', 'price']])
print()

# --- Part 2: Safe modification using .loc ---
# Use .loc to modify the original DataFrame directly
condition = df['stock'] < 50
df.loc[condition, 'price'] = 0.0

print('Original df after safe .loc modification (low-stock prices set to 0.0):')
print(df[['product', 'stock', 'price']])

# Confirm only the correct rows were changed
print('\nVerification — rows with stock < 50 should have price == 0.0:')
print(df.loc[condition, ['product', 'stock', 'price']])


Original DataFrame:
       product category  stock   price
0       Widget        A    150    9.99
1       Gadget        B     30   49.99
2    Doohickey        A    200    4.99
3  Thingamajig        C      5  199.99
4        Gizmo        B     80   29.99

low_stock subset (before attempted modification):
       product category  stock   price
1       Gadget        B     30   49.99
3  Thingamajig        C      5  199.99

No warning raised (behavior may vary by pandas version)

Original df after chained assignment attempt (price should be UNCHANGED):
       product  stock   price
0       Widget    150    9.99
1       Gadget     30   49.99
2    Doohickey    200    4.99
3  Thingamajig      5  199.99
4        Gizmo     80   29.99

Original df after safe .loc modification (low-stock prices set to 0.0):
       product  stock  price
0       Widget    150   9.99
1       Gadget     30   0.00
2    Doohickey    200   4.99
3  Thingamajig      5   0.00
4        Gizmo     80  29.99

Verification — row